## 1. Business Understanding

Every quarter, the ABS publishes CPI and the RBA reacts to it — or, more precisely, to its
own forecast of it, since policy takes months to bite. Three things make this project's
question harder than "fit a time series model":

- **Two targets, not one.** Headline CPI is politically visible; trimmed-mean CPI is what
  the RBA actually watches for the *underlying* inflation trend, since it discards the
  volatile top/bottom 15% of the price-change distribution each quarter.
- **A moving external benchmark.** The RBA publishes its own CPI forecasts in the
  quarterly Statement on Monetary Policy — a credible institutional baseline this project's
  models are expected to compete against, not just a naive seasonal average.
- **Structure vs. accuracy are different jobs.** Predicting *how much* CPI will move
  (SARIMA, Elastic Net, Ensemble) is a different exercise from explaining *why* it moved, or
  what a hypothetical shock would do to it (SVAR, the scenario engine) — conflating the two
  is a common way forecasting projects overstate what they've shown.

**Success criteria**, previewed here so every later section can be read against them:

$$\mathrm{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^n (y_i-\hat y_i)^2}$$

$$P(\text{lower} \le y \le \text{upper}) \approx 0.80$$

$$\text{macro-F1} = \tfrac{1}{3}\sum_{c\in\{cut,hold,hike\}} F1_c$$

point-forecast accuracy per horizon vs. seasonal-naive and the RBA · nominal 80% interval
coverage · the RBA policy-action classifier's class-balanced F1.

**Out of scope.** The SVAR/scenario engine produces *illustrative* structural simulations,
not causal point estimates — both systems fail multivariate residual whiteness and
normality diagnostics even after COVID treatment and a block-bootstrap fix. Every scenario
output downstream is labelled accordingly.

#### Explained, step by step

**RMSE — how far off, on average, in the same units as the data**

$$\mathrm{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^n (y_i - \hat y_i)^2}$$

- $y_i$ is the actual CPI value at test point $i$; $\hat y_i$ is the model's forecast for that same point; $n$ is how many test points are being averaged over (the walk-forward out-of-sample quarters).
- Step by step: subtract the true value from the prediction ($y_i - \hat y_i$) — this is the error, positive or negative. Square it, so every error becomes positive and big misses are punished far more than small ones (an error of 2 contributes 4; an error of 4 contributes 16, not 8). Average the squared errors across all $n$ points, then take the square root to undo the squaring and bring the number back into the original units — percentage points of CPI, not percentage-points-squared. An RMSE of 0.75 means "on average, about 0.75 percentage points off," directly comparable to the CPI numbers themselves.
- Squaring (rather than averaging $|y_i-\hat y_i|$) is what makes a handful of very bad misses show up loudly in the score — appropriate when a badly wrong quarterly inflation call is more costly than being consistently a little off.

**Interval coverage — did the "80% confident" band actually behave like one?**

$$P(\text{lower} \le y \le \text{upper}) \approx 0.80$$

- The model doesn't only output one number — it outputs a range (`lower`, `upper`) meant to contain the true value 80% of the time, across many quarters. This line says: count how many historical test quarters the actual CPI value landed inside the model's predicted interval, divide by the number of quarters — that fraction should sit close to 0.80.
- Analogy: if a forecaster says "80% chance of rain" 100 times and it only rains 40 of those times, their "80%" is meaningless. If the model's 80% band only contains the truth 50% of the time, the interval is too narrow (overconfident); if it contains the truth 98% of the time, it's too wide (underconfident, less useful for decisions). This is exactly why the Ensemble section (4.3) rescales its raw simulated interval before serving it.

**Macro-F1 — one number for a 3-class classifier, without letting the majority class dominate**

$$\text{macro-F1} = \tfrac{1}{3}\sum_{c \in \{cut,hold,hike\}} F1_c$$

- $F1_c$ for one class $c$ is the harmonic mean of *precision* (of the times the model said "cut," how often was it right?) and *recall* (of the times it actually was a cut, how often did the model catch it?) — a model only scores well on $F1_c$ if it's good at both.
- "Hold" is by far the RBA's most common quarterly decision, so a classifier that always predicts "hold" would look deceptively good on plain accuracy. Macro-F1 fixes this by computing $F1_c$ separately for cut, hold, and hike, then averaging the three **unweighted** — so correctly calling a rare "cut" or "hike" counts exactly as much as correctly calling the common "hold." That's why the reportable classifier's headline number is "macro-F1 = 0.775," not plain accuracy.